# Creating files with image-text pairs

This notebook is used to extract image-text pairs we want to retrieve multimodal features from.

Whenever the definition of the target should be used as text, please make sure the mapping file ends with "__definition-form.json"


In [1]:
import json
import os

In [2]:
# some helper functions

def dict_to_file(filepath:str,data:dict,indent:int):
    """
    Saves content of a python dictionary to a json file.
    Parameters:
        filepath (str): filepath specifying where the txt should be saved
        data (dict): data that should be saved to file
        indent (int): indentation representing the spacing used when "writing" into the file
    """
    with open(filepath,"w") as saver_file:
        json.dump(data,saver_file,indent=indent)

def file_to_dict(filepath:str) -> dict:
    """
    Loads json file content into a python dictionary.
    Parameters:
        filepath (str): filepath specifying where the txt should be saved
    Returns:    
        _ (dict): dictionary with content that was saved to file
    """
    with open(filepath) as loader_file: 
        return json.load(loader_file)
    
def tsv_to_dict(filepath:str) -> dict:
    """
    Loads tsv file content into a python dictionary.
    Parameters:
        filepath (str): filepath specifying where the txt should be saved
    Returns:    
        content (dict): dictionary with content that was saved to file
    """
    content = dict()
    with open(filepath) as loader_file: 
        for line in loader_file:
            target, *rest = line.split("\t")
            content[target] = rest
    return content

In [ ]:
mapping_dir = "/path/to/repo/Experiments/Mappings/"

In [ ]:
def create_mapping(image_dir:str,mapping_dir:str,limitation:str=None,definitions:dict=None):
    """
    Creates a json file which represents a list of (img,target) pairs and saves it to file.
    Parameters:
        image_dir (str): path to directory where images are saved
        mapping_dir (str): path to directory where the to be created json file with the mappings should be saved
        limitation (str): (optional) allows to limit the images that are considered in the image_dir
        definitions (dict): (optional) if provided, it considers the values as text, e.g. key = target, value = definition of target
    """
    if os.path.exists(image_dir) and os.path.exists(mapping_dir):
        experiment_identifier = image_dir[:-1].split("Images/")[1].replace("/","__") #"__".join(image_dir.split("/")[-4:-1])
        if limitation: 
            experiment_identifier += "__"+limitation
        if definitions:
            experiment_identifier += "__definition-form"
        mapping_file_path = mapping_dir+image_dir.split("/")[6].replace("Image","").lower()+"__"+experiment_identifier+".json" # note this is specific for the way I generated my images
        image_text_list = []
        for img_file in os.listdir(image_dir):
            # want to extract the target that is shown in the image
            img_identifier = img_file.split(".jpg")[0]
            if img_identifier.startswith("chatgpt_"):
                img_identifier = img_identifier.replace("chatgpt_","")
            target_name = img_identifier.split("_")[0]
            text_prompt = target_name
            if definitions: # use the prompt provided in the definitions dict
                if target_name in definitions:
                    text_prompt = definitions[target_name] 
                else:
                    text_prompt = definitions[target_name+"_"+img_identifier.split("_")[1]]
            if limitation:
                if img_identifier.startswith(target_name+"_"+limitation): # only consider image files that match the limitation
                    image_text_list += [(image_dir+img_file,text_prompt)]    
            else:
                image_text_list += [(image_dir+img_file,text_prompt)]
        # print(len(image_text_list),image_text_list)
        dict_to_file(mapping_file_path,image_text_list,2)
    else:
        print("specified paths don't exist: ",image_dir,os.path.exists(image_dir),mapping_dir,os.path.exists(mapping_dir))

In [ ]:
# create some mappings for images using the according definitions

# load definitions
compound_definitions = tsv_to_dict("/path/to/repo/Data/NounDefinitions/chatgpt_compound_definitions.tsv")
constituent_definitions = tsv_to_dict("/path/to/repo/Data/NounDefinitions/chatgpt_constituent_definitions.tsv")
all_definitions = {target:prompts[0] for definition_dict in [compound_definitions,constituent_definitions] for target,prompts in definition_dict.items()}

image_dirs = [
    "/path/to/repo/Data/Images/ConstituentImages/pixartsigma/chatgpt_constituent_definitions/",
    "/path/to/repo/Data/Images/CompoundImages/noise2img/flux/chatgpt_compound_definition_prompts/direct approach/",
    "/path/to/repo/Data/Images/CompoundImages/head2img/flux/chatgpt_compound_definition_prompts/direct approach/",
    "/path/to/repo/Data/Images/CompoundImages/modifier2img/flux/chatgpt_compound_definition_prompts/direct approach/",
]
limitations = {"CompoundImages":[None], # use all images in the according folders
               "ConstituentImages": [None]} 
for image_dir in image_dirs:
    for approach,image_limitations in limitations.items():
        if approach in image_dir:
            for limit in image_limitations:
                create_mapping(image_dir,mapping_dir,limit,all_definitions)